In [6]:
import re
import os
import csv

input_file = "alcoolismo.txt"

def remover_multiplos_pontos(texto):
    """
    Remove sequências de dois ou mais pontos, substituindo-os por um único espaço.
    Também limpa múltiplos espaços que possam surgir.
    """
    texto_limpo = re.sub(r'\.{2,}', ' ', texto)
    texto_limpo = re.sub(r'\s+', ' ', texto_limpo).strip()
    return texto_limpo

def formatar_sentencas_com_referencias(texto_bruto):
    """
    Formata o texto, garantindo que cada sentença comece com maiúscula e termine com ponto,
    enquanto preserva blocos de referência entre parênteses.
    """
    # Passo 0: Remover múltiplos pontos no início do processo
    texto = remover_multiplos_pontos(texto_bruto)

    # Dicionário para armazenar temporariamente os blocos entre parênteses
    placeholders = {}
    placeholder_counter = 0

    def replace_parentheses_block(match):
        nonlocal placeholder_counter
        key = f"__PLACEHOLDER_{placeholder_counter}__"
        # Substitui quebras de linha e múltiplos espaços DENTRO do bloco por um único espaço
        content = re.sub(r'\s+', ' ', match.group(0))
        placeholders[key] = content
        placeholder_counter += 1
        return key

    # Encontra e substitui blocos entre parênteses por placeholders
    texto_com_placeholders = re.sub(r'\([^\)]*?(?:\n[^\)]*?)*?\)', replace_parentheses_block, texto, flags=re.DOTALL)

    # --- Lógica de Formatação de Sentenças ---
    # Normaliza espaços em branco
    texto_normalizado = re.sub(r'\s+', ' ', texto_com_placeholders).strip()

    # Divide o texto em partes usando qualquer pontuação final de frase como delimitador.
    partes = re.split(r'([.?!])', texto_normalizado)

    sentencas_intermediarias = []
    sentenca_atual = ""

    for i, parte in enumerate(partes):
        parte_limpa = parte.strip()

        if not parte_limpa:
            continue

        if parte_limpa in ['.', '?', '!']:
            if sentenca_atual:
                sentenca_atual += parte_limpa
                sentencas_intermediarias.append(sentenca_atual.strip())
                sentenca_atual = ""
        else:
            if sentenca_atual:
                sentenca_atual += " " + parte_limpa
            else:
                sentenca_atual += parte_limpa

    # Adiciona a última sentença, se houver e não estiver vazia
    if sentenca_atual.strip():
        if not sentenca_atual.strip().endswith(('.', '?', '!')) and not sentenca_atual.strip().endswith('__'):
            sentenca_atual += "."
        sentencas_intermediarias.append(sentenca_atual.strip())

    # --- Pós-processamento: Restaurar os placeholders e aplicar quebras de linha ---
    # Junta as sentenças intermediárias com um espaço.
    texto_reconstruido = " ".join(sentencas_intermediarias)

    # Restaura os placeholders
    for key, value in placeholders.items():
        texto_reconstruido = texto_reconstruido.replace(key, value)

    # Limpeza final de múltiplos espaços que podem ter surgido
    texto_reconstruido = re.sub(r'\s+', ' ', texto_reconstruido).strip()

    # **NOVO MÉTODO PARA ADICIONAR QUEBRAS DE LINHA:**
    # Esta regex busca um ponto final, ponto de interrogação ou exclamação,
    # que NÃO seja seguido imediatamente por um placeholder (look-ahead negativo).
    # E substitui essa pontuação + espaço(s) por ela mesma seguida de uma quebra de linha.
    # Ex: "frase. Outra" -> "frase.\nOutra"
    # O `\g<1>` refere-se ao grupo de captura da pontuação ([.?!]).
    # O `\s*` refere-se a zero ou mais espaços após a pontuação.
    texto_com_quebras = re.sub(r'([.?!])\s*(?!__PLACEHOLDER_\d+__)', r'\g<1>\n', texto_reconstruido)
    
    # Remove quebras de linha duplas/triplas
    texto_com_quebras = re.sub(r'\n+', '\n', texto_com_quebras).strip()

    # Capitaliza a primeira letra de cada linha, se apropriado e não for placeholder
    final_lines_capitalized = []
    for line in texto_com_quebras.split('\n'):
        line_stripped = line.strip()
        if line_stripped: # Garante que a linha não está vazia após o strip
            if not line_stripped.startswith('__PLACEHOLDER_') and line_stripped[0].islower():
                final_lines_capitalized.append(line_stripped[0].upper() + line_stripped[1:])
            else:
                final_lines_capitalized.append(line_stripped)

    return "\n".join(final_lines_capitalized)

def calcular_estatisticas_texto(texto):
    """
    Calcula o número de palavras e o número de sentenças em um texto formatado.
    """
    # Contagem de palavras: divide o texto por espaços e filtra strings vazias
    palavras = texto.split()
    numero_palavras = len(palavras)

    # Contagem de sentenças: cada linha no texto formatado representa uma sentença
    sentencas = [linha for linha in texto.split('\n') if linha.strip()]
    numero_sentencas = len(sentencas)

    return numero_palavras, numero_sentencas

def salvar_estatisticas_csv(numero_palavras, numero_sentencas, nome_arquivo="estatisticas_texto1.csv"):
    """
    Salva as estatísticas do texto em um arquivo CSV.
    """
    try:
        with open(nome_arquivo, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Metrica', 'Valor'])
            writer.writerow(['Numero de Palavras', numero_palavras])
            writer.writerow(['Numero de Sentencas', numero_sentencas])
        print(f"Estatísticas salvas em '{nome_arquivo}'.")
    except Exception as e:
        print(f"Erro ao salvar estatísticas em CSV: {e}")

def dividir_em_arquivos(conteudo_formatado_str, num_arquivos=10, prefixo_saida="saude_p"):
    """
    Divide o conteúdo de texto formatado em um número especificado de arquivos,
    com um número aproximadamente igual de linhas em cada um.
    """
    linhas = [linha for linha in conteudo_formatado_str.split('\n') if linha.strip()] # Garante que as linhas não estejam vazias
    total_linhas = len(linhas)
    
    if total_linhas == 0:
        print("Nenhuma linha para dividir.")
        return

    linhas_por_arquivo = total_linhas // num_arquivos
    linhas_restantes = total_linhas % num_arquivos

    indice_linha_atual = 0
    for i in range(num_arquivos):
        num_linhas_para_este_arquivo = linhas_por_arquivo
        if i < linhas_restantes:
            num_linhas_para_este_arquivo += 1

        nome_arquivo_saida = f"{prefixo_saida}{str(i+1).zfill(00)}.txt" # Changed zfill to 0 for single digit parts and 2 for double
        
        chunk_de_linhas = linhas[indice_linha_atual : indice_linha_atual + num_linhas_para_este_arquivo]
        
        indice_linha_atual += num_linhas_para_este_arquivo

        try:
            with open(nome_arquivo_saida, "w", encoding="utf-8") as f_out:
                f_out.write("\n".join(chunk_de_linhas))
            print(f"Criado: '{nome_arquivo_saida}' com {len(chunk_de_linhas)} linhas.")
        except Exception as e:
            print(f"Erro ao escrever no arquivo '{nome_arquivo_saida}': {e}")


# --- Execução principal do script ---
if __name__ == "__main__":
    try:
        with open(input_file, "r", encoding="utf-8") as f_in:
            conteudo_bruto = f_in.read()

        # 1. Formatar o conteúdo
        conteudo_formatado = formatar_sentencas_com_referencias(conteudo_bruto)

        # 2. Calcular e salvar estatísticas
        num_palavras, num_sentencas = calcular_estatisticas_texto(conteudo_formatado)
        salvar_estatisticas_csv(num_palavras, num_sentencas)

        # 3. Dividir o conteúdo formatado em múltiplos arquivos
        num_partes = 10
        dividir_em_arquivos(conteudo_formatado, num_partes)

        print("\nProcessamento concluído. Verifique os arquivos gerados.")

    except FileNotFoundError:
        print(f"Erro: O arquivo '{input_file}' não foi encontrado. Verifique o caminho.")
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")

Estatísticas salvas em 'estatisticas_texto1.csv'.
Criado: 'saude_p1.txt' com 77 linhas.
Criado: 'saude_p2.txt' com 77 linhas.
Criado: 'saude_p3.txt' com 77 linhas.
Criado: 'saude_p4.txt' com 77 linhas.
Criado: 'saude_p5.txt' com 77 linhas.
Criado: 'saude_p6.txt' com 77 linhas.
Criado: 'saude_p7.txt' com 76 linhas.
Criado: 'saude_p8.txt' com 76 linhas.
Criado: 'saude_p9.txt' com 76 linhas.
Criado: 'saude_p10.txt' com 76 linhas.

Processamento concluído. Verifique os arquivos gerados.


In [5]:
import re
import os
import csv

input_file = "clinica.txt"

def remover_multiplos_pontos(texto):
    """
    Remove sequências de dois ou mais pontos, substituindo-os por um único espaço.
    Também limpa múltiplos espaços que possam surgir.
    """
    texto_limpo = re.sub(r'\.{2,}', ' ', texto)
    texto_limpo = re.sub(r'\s+', ' ', texto_limpo).strip()
    return texto_limpo

def formatar_sentencas_com_referencias(texto_bruto):
    """
    Formata o texto, garantindo que cada sentença comece com maiúscula e termine com ponto,
    enquanto preserva blocos de referência entre parênteses.
    """
    # Passo 0: Remover múltiplos pontos no início do processo
    texto = remover_multiplos_pontos(texto_bruto)

    # Dicionário para armazenar temporariamente os blocos entre parênteses
    placeholders = {}
    placeholder_counter = 0

    def replace_parentheses_block(match):
        nonlocal placeholder_counter
        key = f"__PLACEHOLDER_{placeholder_counter}__"
        # Substitui quebras de linha e múltiplos espaços DENTRO do bloco por um único espaço
        content = re.sub(r'\s+', ' ', match.group(0))
        placeholders[key] = content
        placeholder_counter += 1
        return key

    # Encontra e substitui blocos entre parênteses por placeholders
    texto_com_placeholders = re.sub(r'\([^\)]*?(?:\n[^\)]*?)*?\)', replace_parentheses_block, texto, flags=re.DOTALL)

    # --- Lógica de Formatação de Sentenças ---
    # Normaliza espaços em branco
    texto_normalizado = re.sub(r'\s+', ' ', texto_com_placeholders).strip()

    # Divide o texto em partes usando qualquer pontuação final de frase como delimitador.
    partes = re.split(r'([.?!])', texto_normalizado)

    sentencas_intermediarias = []
    sentenca_atual = ""

    for i, parte in enumerate(partes):
        parte_limpa = parte.strip()

        if not parte_limpa:
            continue

        if parte_limpa in ['.', '?', '!']:
            if sentenca_atual:
                sentenca_atual += parte_limpa
                sentencas_intermediarias.append(sentenca_atual.strip())
                sentenca_atual = ""
        else:
            if sentenca_atual:
                sentenca_atual += " " + parte_limpa
            else:
                sentenca_atual += parte_limpa

    # Adiciona a última sentença, se houver e não estiver vazia
    if sentenca_atual.strip():
        if not sentenca_atual.strip().endswith(('.', '?', '!')) and not sentenca_atual.strip().endswith('__'):
            sentenca_atual += "."
        sentencas_intermediarias.append(sentenca_atual.strip())

    # --- Pós-processamento: Restaurar os placeholders e aplicar quebras de linha ---
    # Junta as sentenças intermediárias com um espaço.
    texto_reconstruido = " ".join(sentencas_intermediarias)

    # Restaura os placeholders
    for key, value in placeholders.items():
        texto_reconstruido = texto_reconstruido.replace(key, value)

    # Limpeza final de múltiplos espaços que podem ter surgido
    texto_reconstruido = re.sub(r'\s+', ' ', texto_reconstruido).strip()

    # **NOVO MÉTODO PARA ADICIONAR QUEBRAS DE LINHA:**
    # Esta regex busca um ponto final, ponto de interrogação ou exclamação,
    # que NÃO seja seguido imediatamente por um placeholder (look-ahead negativo).
    # E substitui essa pontuação + espaço(s) por ela mesma seguida de uma quebra de linha.
    # Ex: "frase. Outra" -> "frase.\nOutra"
    # O `\g<1>` refere-se ao grupo de captura da pontuação ([.?!]).
    # O `\s*` refere-se a zero ou mais espaços após a pontuação.
    texto_com_quebras = re.sub(r'([.?!])\s*(?!__PLACEHOLDER_\d+__)', r'\g<1>\n', texto_reconstruido)
    
    # Remove quebras de linha duplas/triplas
    texto_com_quebras = re.sub(r'\n+', '\n', texto_com_quebras).strip()

    # Capitaliza a primeira letra de cada linha, se apropriado e não for placeholder
    final_lines_capitalized = []
    for line in texto_com_quebras.split('\n'):
        line_stripped = line.strip()
        if line_stripped: # Garante que a linha não está vazia após o strip
            if not line_stripped.startswith('__PLACEHOLDER_') and line_stripped[0].islower():
                final_lines_capitalized.append(line_stripped[0].upper() + line_stripped[1:])
            else:
                final_lines_capitalized.append(line_stripped)

    return "\n".join(final_lines_capitalized)

def calcular_estatisticas_texto(texto):
    """
    Calcula o número de palavras e o número de sentenças em um texto formatado.
    """
    # Contagem de palavras: divide o texto por espaços e filtra strings vazias
    palavras = texto.split()
    numero_palavras = len(palavras)

    # Contagem de sentenças: cada linha no texto formatado representa uma sentença
    sentencas = [linha for linha in texto.split('\n') if linha.strip()]
    numero_sentencas = len(sentencas)

    return numero_palavras, numero_sentencas

def salvar_estatisticas_csv(numero_palavras, numero_sentencas, nome_arquivo="estatisticas_texto2.csv"):
    """
    Salva as estatísticas do texto em um arquivo CSV.
    """
    try:
        with open(nome_arquivo, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Metrica', 'Valor'])
            writer.writerow(['Numero de Palavras', numero_palavras])
            writer.writerow(['Numero de Sentencas', numero_sentencas])
        print(f"Estatísticas salvas em '{nome_arquivo}'.")
    except Exception as e:
        print(f"Erro ao salvar estatísticas em CSV: {e}")

def dividir_em_arquivos(conteudo_formatado_str, num_arquivos=10, prefixo_saida="saude_p"):
    """
    Divide o conteúdo de texto formatado em um número especificado de arquivos,
    com um número aproximadamente igual de linhas em cada um.
    """
    linhas = [linha for linha in conteudo_formatado_str.split('\n') if linha.strip()] # Garante que as linhas não estejam vazias
    total_linhas = len(linhas)
    
    if total_linhas == 0:
        print("Nenhuma linha para dividir.")
        return

    linhas_por_arquivo = total_linhas // num_arquivos
    linhas_restantes = total_linhas % num_arquivos

    indice_linha_atual = 0
    for i in range(num_arquivos):
        num_linhas_para_este_arquivo = linhas_por_arquivo
        if i < linhas_restantes:
            num_linhas_para_este_arquivo += 1

        nome_arquivo_saida = f"{prefixo_saida}{str(i+11).zfill(00)}.txt" # Changed zfill to 0 for single digit parts and 2 for double
        
        chunk_de_linhas = linhas[indice_linha_atual : indice_linha_atual + num_linhas_para_este_arquivo]
        
        indice_linha_atual += num_linhas_para_este_arquivo

        try:
            with open(nome_arquivo_saida, "w", encoding="utf-8") as f_out:
                f_out.write("\n".join(chunk_de_linhas))
            print(f"Criado: '{nome_arquivo_saida}' com {len(chunk_de_linhas)} linhas.")
        except Exception as e:
            print(f"Erro ao escrever no arquivo '{nome_arquivo_saida}': {e}")


# --- Execução principal do script ---
if __name__ == "__main__":
    try:
        with open(input_file, "r", encoding="utf-8") as f_in:
            conteudo_bruto = f_in.read()

        # 1. Formatar o conteúdo
        conteudo_formatado = formatar_sentencas_com_referencias(conteudo_bruto)

        # 2. Calcular e salvar estatísticas
        num_palavras, num_sentencas = calcular_estatisticas_texto(conteudo_formatado)
        salvar_estatisticas_csv(num_palavras, num_sentencas)

        # 3. Dividir o conteúdo formatado em múltiplos arquivos
        num_partes = 10
        dividir_em_arquivos(conteudo_formatado, num_partes)

        print("\nProcessamento concluído. Verifique os arquivos gerados.")

    except FileNotFoundError:
        print(f"Erro: O arquivo '{input_file}' não foi encontrado. Verifique o caminho.")
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")


Estatísticas salvas em 'estatisticas_texto2.csv'.
Criado: 'saude_p11.txt' com 204 linhas.
Criado: 'saude_p12.txt' com 204 linhas.
Criado: 'saude_p13.txt' com 204 linhas.
Criado: 'saude_p14.txt' com 204 linhas.
Criado: 'saude_p15.txt' com 204 linhas.
Criado: 'saude_p16.txt' com 204 linhas.
Criado: 'saude_p17.txt' com 204 linhas.
Criado: 'saude_p18.txt' com 204 linhas.
Criado: 'saude_p19.txt' com 204 linhas.
Criado: 'saude_p20.txt' com 203 linhas.

Processamento concluído. Verifique os arquivos gerados.
